In [1]:
import numpy as np
import h5py
import torch
import torch.nn as nn
import random
import time
from tqdm import tqdm
import os
from matplotlib import font_manager as fm
import matplotlib.pyplot as plt

# ==================== 0. 环境与字体设置 ====================
def set_chinese_font():
    try:
        font_paths = ['/usr/share/fonts/truetype/wqy/wqy-microhei.ttc',
                      'C:/Windows/Fonts/simhei.ttf',
                      '/System/Library/Fonts/PingFang.ttc']
        for font_path in font_paths:
            if os.path.exists(font_path):
                fm.fontManager.addfont(font_path)
                plt.rcParams['font.family'] = fm.FontProperties(fname=font_path).get_name()
                plt.rcParams['axes.unicode_minus'] = False
                return True
    except: pass
    return False
set_chinese_font()

# ==================== 1. 卫星抗干扰仿真环境 (保持一致) ====================
class SatelliteEnvV3:
    def __init__(self, h5_path):
        print(f"📂 正在预载入数据集: {os.path.basename(h5_path)} ...")
        with h5py.File(h5_path, 'r') as f:
            self.pred_map = torch.FloatTensor(f['Y_horizon'][:]) 
            self.truth_map = torch.FloatTensor(f['Y_horizon'][:])
            self.type_data = torch.FloatTensor(f['gt_type'][:])
            
        self.num_channels = 10
        self.max_steps = len(self.pred_map)
        self.current_step = 0
        self.last_action = 0

    def reset(self):
        self.current_step = 0
        self.last_action = random.randint(0, 9)
        return self._get_state()

    def _get_state(self):
        # 贪心搜索需要查看当前的预测热图
        map_feat = self.pred_map[self.current_step] # (10, 10) -> 未来10步，10个信道
        return map_feat

    def step(self, action):
        is_collision = self.truth_map[self.current_step, 0, action] > 0.5
        future_risk = torch.mean(self.pred_map[self.current_step, :, action])
        
        if is_collision:
            reward = -100.0
        else:
            reward = 15.0 - (future_risk.item() * 30.0)
            if action == self.last_action:
                reward += 5.0 
            else:
                reward -= 2.0 
            
        self.last_action = action
        self.current_step += 1
        done = self.current_step >= self.max_steps - 1
        next_state = self._get_state() if not done else torch.zeros((10, 10))
        return next_state, reward, done, is_collision

# ==================== 2. 贪心搜索智能体 (Greedy Agent) ====================
class GreedySearchAgent:
    def __init__(self, action_dim=10):
        self.action_dim = action_dim

    def choose_action(self, state):
        """
        核心逻辑：在 Transformer 预测图中，寻找未来第1步（Index 0）概率最低的信道
        """
        start_time = time.time()
        
        # state 形状为 (10, 10), 第一维是未来步长，第二维是信道
        current_step_prediction = state[0] # 只看最近的一个预测步长
        
        # 🟢 贪心策略：选择当前时刻被干扰概率最小的信道
        action = torch.argmin(current_step_prediction).item()
        
        latency = time.time() - start_time
        return action, latency

# ==================== 3. 运行评估程序 (与 DDQN 输出对齐) ====================
def run_greedy_search_evaluation():
    DATA_PATH = "/root/autodl-tmp/validate/0218/Prediction/sim_dataset_v6_5/academic_long_horizon_v6_5_200k.h5"
    env = SatelliteEnvV3(DATA_PATH)
    agent = GreedySearchAgent()
    
    episodes = 100
    metrics = {'reward': [], 'sr': [], 'hops': [], 'latency': [], 'throughput': [], 'stability': []}

    print("🚀 启动贪心搜索 (Greedy Search) 性能评估...")
    for ep in range(episodes):
        state = env.reset()
        ep_reward, collisions, steps, hops = 0, 0, 0, 0
        ep_latencies, actions = [], []

        pbar = tqdm(total=env.max_steps, desc=f"Ep {ep+1}/{episodes}", leave=False)
        while True:
            action, lat = agent.choose_action(state)
            ep_latencies.append(lat)
            actions.append(action)

            if steps > 0 and action != env.last_action:
                hops += 1
                
            next_state, reward, done, collision = env.step(action)
            
            state = next_state
            ep_reward += reward
            if collision: collisions += 1
            steps += 1
            pbar.update(1)
            if done: break
        
        pbar.close()
        
        # 指标计算
        sr = (1 - collisions / steps) * 100
        avg_hops = hops / (steps / 100)
        avg_lat = np.mean(ep_latencies) * 1000
        throughput = (1 - (collisions / steps)) * 1.0
        stability = np.std(actions)

        metrics['reward'].append(ep_reward)
        metrics['sr'].append(sr)
        metrics['hops'].append(avg_hops)
        metrics['latency'].append(avg_lat)
        metrics['throughput'].append(throughput)
        metrics['stability'].append(stability)

        if (ep + 1) % 10 == 0:
            print(f"✅ 已完成 {ep+1} 轮贪心实验 | 当前平均成功率: {np.mean(metrics['sr']):.2f}%")

    # --- 最终对比报告 ---
    print("\n" + "="*50)
    print("📊 贪心搜索 (Greedy Search) 实验报告总结")
    print("-" * 50)
    print(f"1. 收敛轮次 (Convergence Episode): N/A (启发式方法)")
    print(f"2. 平均推理时延 (Inference Latency): {np.mean(metrics['latency']):.4f} ms")
    print(f"3. 避障成功率平均值 (Avg Success Rate): {np.mean(metrics['sr']):.2f} %")
    print(f"4. 稳态跳频代价 (Avg Switching Cost): {np.mean(metrics['hops']):.2f} hops/100steps")
    print(f"5. 归一化吞吐量 (Throughput): {np.mean(metrics['throughput']):.4f}")
    print(f"6. 动作稳定性 (Policy Stability Std): {np.mean(metrics['stability']):.4f}")
    print("="*50)

    return metrics

if __name__ == "__main__":
    run_greedy_search_evaluation()

📂 正在预载入数据集: academic_long_horizon_v6_5_200k.h5 ...
🚀 启动贪心搜索 (Greedy Search) 性能评估...


✅ 已完成 10 轮贪心实验 | 当前平均成功率: 100.00%


✅ 已完成 20 轮贪心实验 | 当前平均成功率: 100.00%


✅ 已完成 30 轮贪心实验 | 当前平均成功率: 100.00%


✅ 已完成 40 轮贪心实验 | 当前平均成功率: 100.00%


...

✅ 已完成 60 轮贪心实验 | 当前平均成功率: 100.00%


✅ 已完成 70 轮贪心实验 | 当前平均成功率: 100.00%


✅ 已完成 80 轮贪心实验 | 当前平均成功率: 100.00%


✅ 已完成 90 轮贪心实验 | 当前平均成功率: 100.00%


✅ 已完成 100 轮贪心实验 | 当前平均成功率: 100.00%

📊 贪心搜索 (Greedy Search) 实验报告总结
--------------------------------------------------
1. 收敛轮次 (Convergence Episode): N/A (启发式方法)
2. 平均推理时延 (Inference Latency): 0.0050 ms
3. 避障成功率平均值 (Avg Success Rate): 100.00 %
4. 稳态跳频代价 (Avg Switching Cost): 2.19 hops/100steps
5. 归一化吞吐量 (Throughput): 1.0000
6. 动作稳定性 (Policy Stability Std): 0.5871
